In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, coalesce, lit, expr, split, lpad, concat,
    sum as spark_sum, avg, median, percentile_approx,
    initcap
)
from pyspark.sql.window import Window


# TASK-2: FIX NULL COST_PRICE IN PRODUCTS TABLE

# Business Logic:
# - Learn company margin from valid rows: margin = (unit_price - cost_price) / cost_price
# - Reverse engineer: cost_price = unit_price / (1 + margin)
# - Use category-specific margins when available


print("Loading tables for cost_price imputation...\n")

# ---------------- LOAD TABLES ----------------
ili = spark.table("bronze_catalog.bronze_stage_schema.invoice_line_items")
products = spark.table("bronze_catalog.bronze_stage_schema.products")
invoices = spark.table("bronze_catalog.bronze_stage_schema.invoices")
fx = spark.table("bronze_catalog.bronze_raw_schema.exchange_rates")



# ---------------- PREPARE DATES ----------------
# Parse invoice_date from yyyy/M/d format
invoices = invoices.withColumn("inv_date_parts", split(col("invoice_date"), "/"))
invoices = invoices.withColumn(
    "invoice_date_normalized",
    concat(
        col("inv_date_parts")[0],
        lit("-"),
        lpad(col("inv_date_parts")[1], 2, "0"),
        lit("-"),
        lpad(col("inv_date_parts")[2], 2, "0")
    )
).withColumn(
    "invoice_date_clean",
    expr("to_date(invoice_date_normalized, 'yyyy-MM-dd')")
)

# Parse exchange rate date
fx = fx.withColumn("fx_date_parts", split(col("date"), "/"))
fx = fx.withColumn(
    "date_normalized",
    concat(
        col("fx_date_parts")[0],
        lit("-"),
        lpad(col("fx_date_parts")[1], 2, "0"),
        lit("-"),
        lpad(col("fx_date_parts")[2], 2, "0")
    )
).withColumn(
    "date_clean",
    expr("to_date(date_normalized, 'yyyy-MM-dd')")
)

# ---------------- JOIN TABLES ----------------
# Start with invoice line items
df = ili.select("invoice_id", "product_id", "unit_price")

# Join invoices for currency and date
df = df.join(
    invoices.select("invoice_id", "currency", "invoice_date_clean"),
    "invoice_id",
    "left"
)

# Join products for category and cost_price
products_clean = products.withColumn(
    "category",
    initcap(coalesce(col("category"), lit("Unknown")))
)

df = df.join(
    products_clean.select("product_id", "product_name", "category", "cost_price"),
    "product_id",
    "left"
)

# Join exchange rates for FX conversion
df = df.join(
    fx.select("date_clean", col("currency").alias("fx_currency"), "rate_to_usd"),
    (col("invoice_date_clean") == col("date_clean")) & (col("currency") == col("fx_currency")),
    "left"
)


display(df.select("product_id", "product_name", "category", "unit_price", "cost_price", "currency", "rate_to_usd").limit(15))

In [0]:

# STANDARDIZE PRICES TO USD

# Handle multiple currencies by converting all prices to USD
# This ensures consistent margin calculations across currencies

print(" Converting prices to USD for consistent margin analysis...\n")

# Convert unit_price to USD
df = df.withColumn(
    "unit_price_usd",
    col("unit_price") * coalesce(col("rate_to_usd"), lit(1.0))
)

# Convert cost_price to USD
df = df.withColumn(
    "cost_price_usd",
    col("cost_price") * coalesce(col("rate_to_usd"), lit(1.0))
)

print(" Price standardization complete")

# Check NULL cost_price statistics
null_cost_stats = df.groupBy(
    when(col("cost_price").isNull(), "NULL cost_price")
    .otherwise("Has cost_price")
    .alias("cost_status")
).agg(
    expr("count(*)").alias("row_count"),
    expr("count(distinct product_id)").alias("unique_products")
)

print("\nNULL Cost Price Statistics:")
display(null_cost_stats)

# Show currency distribution
currency_dist = df.groupBy("currency").agg(
    expr("count(*)").alias("transactions"),
    expr("avg(rate_to_usd)").alias("avg_fx_rate")
).orderBy(col("transactions").desc())

print("\nCurrency Distribution:")
display(currency_dist)

print("\n Sample of USD-standardized prices:")
display(
    df.select(
        "product_id",
        "category",
        "currency",
        "unit_price",
        "unit_price_usd",
        "cost_price",
        "cost_price_usd"
    ).limit(15)
)

In [0]:

# LEARN COMPANY MARGIN

# Calculate margin from valid rows:
# margin = (unit_price_usd - cost_price_usd) / cost_price_usd

print(" Calculating company margins from valid data...\n")

# Filter to valid rows only (non-NULL prices and positive cost)
valid = df.filter(
    col("unit_price_usd").isNotNull() &
    col("cost_price_usd").isNotNull() &
    (col("cost_price_usd") > 0)
)

print(f" Valid rows for margin calculation: {valid.count():,}")

# Calculate margin
valid = valid.withColumn(
    "margin",
    (col("unit_price_usd") - col("cost_price_usd")) / col("cost_price_usd")
)

# Remove extreme outliers (margin < 0 or > 2.0 means 200%)
print("\n Removing outliers (margin < 0 or > 2.0)...")
valid_clean = valid.filter(
    (col("margin") > 0) & (col("margin") < 2.0)
)

outliers_removed = valid.count() - valid_clean.count()
print(f"  Removed {outliers_removed} outlier records")
print(f"Clean margin records: {valid_clean.count():,}")

# Calculate GLOBAL median margin
global_margin_value = valid_clean.approxQuantile("margin", [0.5], 0.01)[0]

print(f"\n GLOBAL MEDIAN MARGIN: {global_margin_value:.4f} ({global_margin_value*100:.2f}%)")

# Calculate CATEGORY-LEVEL median margins
category_margins = valid_clean.groupBy("category").agg(
    expr("percentile_approx(margin, 0.5)").alias("category_margin"),
    expr("count(*)").alias("sample_size"),
    expr("avg(margin)").alias("avg_margin")
).orderBy("category")

print("\n CATEGORY-LEVEL MARGINS:")
display(category_margins)

# Store global margin as a broadcast variable for later use
from pyspark.sql.functions import lit
global_margin = lit(global_margin_value)

print(f"\n Margin analysis complete!")
print(f"   • Global Margin: {global_margin_value*100:.2f}%")
print(f"   • Will use category-specific margins where available")
print(f"   • Will fallback to global margin for unknown categories")

# Show margin distribution
print("\nMargin Distribution by Category and Product:")
display(
    valid_clean.groupBy("category", "product_id", "product_name").agg(
        expr("percentile_approx(margin, 0.5)").alias("median_margin"),
        expr("count(*)").alias("transactions")
    ).orderBy("category", "product_id").limit(20)
)

In [0]:
# =====================================================
# IMPUTE MISSING COST PRICE
# =====================================================
# Reverse engineer cost_price from unit_price using learned margin:
# cost_price = unit_price / (1 + margin)

print(" Imputing missing cost_price values...\n")

# Join category margins to the main dataframe
df_with_margins = df.join(
    category_margins.select("category", col("category_margin")),
    "category",
    "left"
)

# Derive cost_price using the following logic:
# 1. If cost_price exists -> keep it
# 2. If cost_price is NULL:
#    a. If unit_price is NULL -> leave as NULL
#    b. Otherwise, use: unit_price / (1 + margin)
#       - Use category_margin if available
#       - Otherwise use global_margin

df_imputed = df_with_margins.withColumn(
    "cost_price_imputed",
    when(
        col("cost_price").isNotNull(),
        col("cost_price")  # Keep original
    ).when(
        col("unit_price").isNull(),
        lit(None)  # Can't derive without unit_price
    ).otherwise(
        # Derive from unit_price using margin
        col("unit_price") / (1 + coalesce(col("category_margin"), global_margin))
    )
)

# Add a flag to track imputed values
df_imputed = df_imputed.withColumn(
    "cost_price_source",
    when(col("cost_price").isNotNull(), "original")
    .when(col("cost_price_imputed").isNotNull(), "imputed_from_margin")
    .otherwise("missing_unit_price")
)

print(" Cost price imputation complete!")

# Statistics
imputation_stats = df_imputed.groupBy("cost_price_source").agg(
    expr("count(*)").alias("row_count"),
    expr("count(distinct product_id)").alias("unique_products")
).orderBy("cost_price_source")

print("\n IMPUTATION STATISTICS:")
display(imputation_stats)

# Show before/after comparison
print("\n Before/After Comparison (Products with NULL cost_price):")
comparison = df_imputed.filter(
    col("cost_price").isNull() & col("cost_price_imputed").isNotNull()
).select(
    "product_id",
    "product_name",
    "category",
    "unit_price",
    col("cost_price").alias("original_cost"),
    col("cost_price_imputed").alias("imputed_cost"),
    col("category_margin"),
    "cost_price_source"
).distinct().orderBy("product_id")

display(comparison.limit(20))

# Verify by category
print("\n Imputation Summary by Category:")
category_summary = df_imputed.groupBy("category", "cost_price_source").agg(
    expr("count(distinct product_id)").alias("products")
).orderBy("category", "cost_price_source")

display(category_summary)

In [0]:
# =====================================================
# FINAL PRODUCTS TABLE
# =====================================================
# Create clean products table with unique products
# Include quality tracking columns

print(" Creating final products table...\n")

# Select relevant columns and get unique products
products_clean = df_imputed.select(
    "product_id",
    "product_name",
    "category",
    col("cost_price_imputed").cast("int").alias("cost_price"),  # Cast to integer
    "cost_price_source"
).dropDuplicates(["product_id"])

# Sort by product_id for readability
products_clean = products_clean.orderBy("product_id")


print("\n FINAL COST PRICE STATUS:")
final_stats = products_clean.groupBy("cost_price_source").agg(
    expr("count(*)").alias("product_count")
).orderBy("cost_price_source")
display(final_stats)

print("\n PRODUCTS BY CATEGORY:")
category_count = products_clean.groupBy("category").agg(
    expr("count(*)").alias("product_count"),
    expr("sum(case when cost_price_source = 'imputed_from_margin' then 1 else 0 end)").alias("imputed_count")
).orderBy("category")
display(category_count)

print("\n FINAL PRODUCTS TABLE (All 60 Products):")
display(products_clean)

# Summary statistics
print("\nCOST PRICE STATISTICS:")
cost_stats = products_clean.filter(col("cost_price").isNotNull()).select(
    expr("min(cost_price)").alias("min_cost"),
    expr("max(cost_price)").alias("max_cost"),
    expr("avg(cost_price)").alias("avg_cost"),
    expr("percentile_approx(cost_price, 0.5)").alias("median_cost")
)
display(cost_stats)

In [0]:

# SAVE TO SILVER CATALOG


print(" Saving products table to Silver Catalog...\n")

# Define target table path
target_table = "silver_catalog.silver_schema.products"

try:
    # Drop existing table to avoid schema merge conflicts
    spark.sql(f"DROP TABLE IF EXISTS {target_table}")
    
    # Write to silver catalog
    products_clean.write.mode("overwrite").saveAsTable(target_table)
  
    
    # Verify the save
  
    saved_table = spark.table(target_table)
    print(f"Verified: {saved_table.count():,} records in silver catalog")
    
    # Display table schema
  
    saved_table.printSchema()
  
    
    print("\n SCHEMA SUMMARY:")
  
    for field in saved_table.schema.fields:
        nullable = "nullable" if field.nullable else "not null"
        print(f"  {field.name:30s} | {str(field.dataType):20s} | {nullable}")

    
    print("\n Table Columns:")
    for col_name in saved_table.columns:
        print(f"   • {col_name}")
    
    
    
except Exception as e:
    print(f" ERROR: Failed to save table")
    print(f"Error details: {str(e)}")
    raise